[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanenalmayouf/applied-ml-workshop/blob/main/labs_colab/day4/lab_4_manafeth_evaluation.ipynb)

<div dir="rtl">

# 🔍 مختبر اليوم الرابع — مقارنة نماذج المغادرة وتحليل الأخطاء

**ورشة أسس تعلم الآلة التطبيقي — اليوم 4 من 5**

هذا الدفتر مبني على مواصفات مختبرات «منافذ» المعتمدة للدورة، ومُجهَّز للعمل مباشرة في **Google Colab** أو في Jupyter محليًا.

**كيف تفتحه في Colab:** اضغط زر «Open in Colab» أعلى هذا الدفتر (أو أعلى الـ README المرافق له). ستُحمَّل بيانات هذا اليوم تلقائيًا من مستودع الدورة على GitHub بمجرد تشغيل خلية إعداد البيانات — بلا أي رفع يدوي. فقط إذا تعذّر الاتصال بالإنترنت داخل Colab لأي سبب، ستظهر خانة احتياطية لرفع ملف `manafeth_data_package.zip` يدويًا.

> 📌 راجع `00_start_here.md` قبل البدء لمعرفة طريقة استخدام خلايا **فكّر أولًا** و**TODO** و**مساعدة** في هذه الدفاتر.

</div>

In [ ]:
from pathlib import Path
import pandas as pd

# 1) نبحث عن مجلد البيانات بجانب هذا الدفتر (يعمل محليًا، أو في Colab إذا رفعت المجلد كاملًا)
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data"), Path("../data/raw"), Path("data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)

# 2) إذا لم نجد المجلد ونحن داخل Google Colab (فتحت الدفتر بزر Open in Colab ولم يُرفَق مجلد البيانات
#    تلقائيًا)، نحمّل حزمة بيانات هذا اليوم مباشرة من نفس المستودع على GitHub — بلا أي تدخل منك
if DATA_DIR is None:
    try:
        import google.colab  # يفشل الاستيراد إن لم نكن داخل Colab، وننتقل إلى except أدناه
        import urllib.request
        import zipfile

        zip_url = "https://raw.githubusercontent.com/hanenalmayouf/applied-ml-workshop/main/labs_colab/day4/manafeth_data_package.zip"
        zip_name = "manafeth_data_package.zip"
        print("لم يتم العثور على مجلد البيانات محليًا — يجري تحميلها تلقائيًا من مستودع الدورة على GitHub...")
        try:
            urllib.request.urlretrieve(zip_url, zip_name)
            with zipfile.ZipFile(zip_name) as z:
                z.extractall(".")
            DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)
            if DATA_DIR is not None:
                print("تم تحميل البيانات تلقائيًا بنجاح ✅")
        except Exception as download_error:
            print("تعذّر التحميل التلقائي من GitHub:", download_error)

        # خطة بديلة فقط إذا فشل التحميل التلقائي (مثلًا بلا اتصال إنترنت داخل Colab)
        if DATA_DIR is None:
            from google.colab import files

            print("سنطلب منك رفع حزمة البيانات يدويًا كخطة بديلة — ارفع ملف manafeth_data_package.zip المرفق مع هذا المختبر.")
            uploaded = files.upload()
            for name in uploaded:
                if name.lower().endswith(".zip"):
                    with zipfile.ZipFile(name) as z:
                        z.extractall(".")
            DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), Path("manafeth_data_package"))
    except ImportError:
        DATA_DIR = Path("manafeth_data_package")
        print("تنبيه: لسنا داخل Google Colab ولم يوجد مجلد بيانات — ضع حزمة البيانات بجانب الدفتر.")

CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"

print("مجلد البيانات المستخدم:", DATA_DIR.resolve())
assert CUSTOMERS_PATH.exists(), "تعذّر العثور على manafeth_customers.parquet — تأكد من رفع حزمة البيانات كاملة."

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (7, 4)

<div dir="rtl">

## 🎯 هدف المختبر

تقارن اليوم نموذجًا مبدئيًا بنماذج شجرية على مشكلة مغادرة العملاء الحقيقية، وتختار مقياس تقييم يناسب فئة إيجابية أقل ظهورًا، ثم تحلّل صفوف الخطأ بدل الاحتفال برقم واحد.

## السيناريو والبيانات

نسبة `churned_30d = 1` في حزمة منافذ نحو **14%** فقط — لذلك لا تكفي الدقة (accuracy) وحدها. نستخدم **متوسط الدقة (Average Precision)** الذي يلخّص منحنى الدقة–الاستدعاء، والاستدعاء بين أعلى 20% من العملاء ترتيبًا (يناسب قائمة متابعة محدودة لفريق خدمة العملاء).

| النموذج | دوره | إعداد مبدئي |
|---|---|---|
| الانحدار اللوجستي | خط أساس قابل للتفسير | `max_iter=1000` |
| شجرة القرار | نموذج أسئلة متتابعة بسيط | `max_depth=4` |
| الغابة العشوائية | تجميع أشجار للتصويت | `n_estimators=150` |
| XGBoost | تعزيز متدرج للمقارنة | `n_estimators=100`، `max_depth=3` |

</div>

<div dir="rtl">

## 🤔 فكّر أولًا

لماذا لا نستخدم بيانات الاختبار لاختيار أفضل نموذج من الجدول، ونحتفظ بها فقط لتقييم النموذج المختار مرة واحدة في النهاية؟

</div>

In [ ]:
customers = pd.read_parquet(CUSTOMERS_PATH)

safe_features = [
    "city", "city_tier", "device", "payment_method",
    "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating", "last_promo_used"
]
numeric_features = [
    "city_tier", "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating"
]
categorical_features = ["city", "device", "payment_method", "last_promo_used"]

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

X = customers[safe_features]
y = customers["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ("numbers", Pipeline([
        ("fill", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler())
    ]), numeric_features),
    ("categories", Pipeline([
        ("fill", SimpleImputer(strategy="constant", fill_value="غير_معروف")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    ConfusionMatrixDisplay, recall_score
)
from xgboost import XGBClassifier

models = {
    "انحدار لوجستي": LogisticRegression(max_iter=1000),
    "شجرة قرار": DecisionTreeClassifier(max_depth=4, random_state=42),
    "غابة عشوائية": RandomForestClassifier(n_estimators=150, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=100, max_depth=3,
        eval_metric="logloss", random_state=42
    )
}

comparison = []
for name, model in models.items():
    # TODO: ابنِ workflow = Pipeline([("prepare", preprocessor), ("model", model)])
    workflow = ...
    # TODO: احسب scores بـ cross_val_score على X_train/y_train بـ cv=5 وscoring="average_precision"
    scores = ...
    comparison.append([name, scores.mean(), scores.std()])

results_df = pd.DataFrame(
    comparison,
    columns=["النموذج", "متوسط الدقة", "تغير النتيجة بين الطيات"]
).sort_values("متوسط الدقة", ascending=False)
results_df

<div dir="rtl">

### 📊 رسم: مقارنة متوسط الدقة بين النماذج

مثال مُنفَّذ — تحويل جدول `results_df` إلى مخطط أعمدة لمقارنة بصرية سريعة.

</div>

In [ ]:
plt.bar(results_df["النموذج"], results_df["متوسط الدقة"], color="#29BA74")
plt.title("متوسط الدقة (Average Precision) بالتحقق المتقاطع")
plt.ylabel("متوسط الدقة")
plt.xticks(rotation=20)
plt.show()

<div dir="rtl">

## اختيار النموذج

اختر النموذج المرشح بسبب مكتوب: المقياس أولًا، ثم قابلية الفهم أو استقرار النتيجة إن كانت الفروق صغيرة.

</div>

In [ ]:
chosen_name = results_df.iloc[0]["النموذج"]
print("النموذج المختار:", chosen_name)
final_workflow = Pipeline([
    ("prepare", preprocessor),
    ("model", models[chosen_name])
])
final_workflow.fit(X_train, y_train)
probabilities = final_workflow.predict_proba(X_test)[:, 1]

ap = average_precision_score(y_test, probabilities)
precision, recall, _ = precision_recall_curve(y_test, probabilities)
plt.plot(recall, precision, color="#5B4FCF")
plt.xlabel("الاستدعاء")
plt.ylabel("الإحكام")
plt.title("منحنى الدقة–الاستدعاء")
plt.show()

contact_count = int(len(y_test) * 0.20)
top_customers = pd.DataFrame({
    "الحقيقة": y_test.to_numpy(),
    "احتمال_المغادرة": probabilities
}).sort_values("احتمال_المغادرة", ascending=False).head(contact_count)
recall_at_20 = top_customers["الحقيقة"].sum() / y_test.sum()
print("متوسط الدقة على الاختبار:", round(ap, 3))
print("الاستدعاء بين أعلى 20%:", round(recall_at_20, 3))

<div dir="rtl">

### 📊 مصفوفة الالتباس عند عتبة 0.50

المواصفة تستورد `ConfusionMatrixDisplay` لكنها لم تكن تستدعيها فعليًا — هنا نضيف الاستدعاء الفعلي حتى ترى شكل الأخطاء الأربعة (صحيح موجب/سالب، خطأ موجب/سالب).

</div>

In [ ]:
# TODO: احسب predictions_at_50 من probabilities عند عتبة 0.50 (استخدم >= ثم .astype(int))
predictions_at_50 = ...

# TODO: استدعِ ConfusionMatrixDisplay.from_predictions(y_test, predictions_at_50, ...) وأضف عنوانًا

<div dir="rtl">

## تحليل الأخطاء

استخرج خمسة عملاء أخطأ النموذج في تصنيفهم عند العتبة 0.50.

</div>

In [ ]:
review = X_test.copy()
review["الحقيقة"] = y_test.to_numpy()
review["احتمال_المغادرة"] = probabilities
review["التوقع_عند_0_50"] = predictions_at_50
errors = review[review["الحقيقة"] != review["التوقع_عند_0_50"]]
errors.head()

<div dir="rtl">

## النتيجة المتوقعة

جدول مقارنة لأربعة نماذج، ومخطط أعمدة لمقارنتها، ومنحنى دقة–استدعاء، ومصفوفة التباس فعلية، وقيمتا متوسط الدقة والاستدعاء بين أعلى 20%. **لا تكتب رقمًا ثابتًا للأداء في أي عرض تقديمي** — قد تختلف النتائج بحسب الإصدار والإعدادات. المهم أن تبرر اختيارك ببيانات التدريب وتشرح أين تظهر حالات التفويت في مصفوفة الالتباس.

## المهارات التي راجعتها

اختيار مقياس يناسب عدم توازن الفئات، استخدام التحقق المتقاطع، مقارنة خطوط معالجة متكافئة، قراءة منحنى الدقة–الاستدعاء ومصفوفة الالتباس، وتحليل أخطاء فعلية.

</div>

<div dir="rtl">

## ✅ تحقق ذاتيًا قبل إغلاق الدفتر
- [ ] `results_df` يحتوي على أربعة نماذج ومتوسط دقة لكل منها
- [ ] اخترت النموذج **قبل** فتح بيانات الاختبار، بسبب مكتوب
- [ ] عرضت مصفوفة الالتباس الفعلية عند عتبة 0.50 (وليس رقمًا مستوردًا فقط)
- [ ] راجعت خمسة صفوف خطأ فعلية

**غدًا:** تجمع كل هذا في مشروع متكامل، وتفحص كيف يتصرف النموذج على شهر لاحق من دون هدف معروف.

</div>